In [ ]:
from google.colab import drive
drive.mount("content/drive")

In [ ]:
import logging
import warnings
import pickle
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, classification_report
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger(__name__)

# CONFIG

In [ ]:
class ModelConfig:
    BASE_DIR    = Path("/content/drive/MyDrive/IDX_Data")
    PROC_DIR    = BASE_DIR / "processed"
    SPLIT_DIR   = BASE_DIR / "pipeline_output"
    MODEL_DIR   = BASE_DIR / "models" / "pretrained"

    TARGET_COL  = "label"
    FEATURE_COLS = [
        "ret_1d", "ret_5d", "ret_20d",
        "rsi_14", "macd_hist", "bb_pct", "bb_width",
        "volatility_20d", "volume_ratio", "price_vs_sma20",
        "intraday_range",
        "usdidr", "brent", "coal", "bi_rate",
        "sentiment_score",
    ]

    # XGBoost / LGBM params
    XGBOOST_PARAMS = {
        "n_estimators": 500,
        "max_depth": 6,
        "learning_rate": 0.05,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "use_label_encoder": False,
        "eval_metric": "logloss",
        "tree_method": "gpu_hist",   # GPU acceleration
        "random_state": 42,
    }

    LGBM_PARAMS = {
        "n_estimators": 500,
        "max_depth": 7,
        "learning_rate": 0.05,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "device": "gpu",
        "random_state": 42,
        "verbose": -1,
    }

    # Ensemble weights
    ENSEMBLE_WEIGHTS = {"xgboost": 0.4, "lgbm": 0.4, "tabnet": 0.2}

    def __post_init__(self):
        self.MODEL_DIR.mkdir(parents=True, exist_ok=True)


cfg = ModelConfig()
cfg.MODEL_DIR.mkdir(parents=True, exist_ok=True)


# DATA LOADER FOR MODELS

In [ ]:
class ModelDataLoader:
    """Load and validate feature datasets from pipeline output."""

    def __init__(self):
        self.scaler = StandardScaler()

    def load_splits(self, dataset_path: Optional[Path] = None) -> Dict[str, pd.DataFrame]:
        """Load train/val/test from parquet or CSV."""
        if dataset_path and dataset_path.exists():
            df = pd.read_parquet(dataset_path)
        else:
            # Reconstruct from ohlcv processed
            logger.warning("No prebuilt dataset found — loading from processed OHLCV.")
            ohlcv_path = cfg.PROC_DIR / "ohlcv_processed.parquet"
            if not ohlcv_path.exists():
                raise FileNotFoundError("Run 01_data_loader.py first.")
            df = pd.read_parquet(ohlcv_path)
            df = self._build_labels(df)

        df["date"] = pd.to_datetime(df["date"])
        train = df[df["date"] <= "2022-12-31"].copy()
        val   = df[(df["date"] > "2022-12-31") & (df["date"] <= "2023-12-31")].copy()
        test  = df[df["date"] > "2023-12-31"].copy()

        logger.info(f"Splits — Train: {len(train):,} | Val: {len(val):,} | Test: {len(test):,}")
        return {"train": train, "val": val, "test": test}

    def _build_labels(self, df: pd.DataFrame, forward_days: int = 5) -> pd.DataFrame:
        """
        Build forward return labels PER TICKER — strictly event-driven.
        Label = 1 if ret_{forward_days}d > 0 else 0.
        IMPORTANT: uses only per-ticker future data, not group-level look-ahead.
        """
        df = df.sort_values(["ticker", "date"]).copy()
        results = []
        for ticker, grp in df.groupby("ticker"):
            g = grp.copy().reset_index(drop=True)
            g["future_ret"] = g["adj_close"].shift(-forward_days) / g["adj_close"] - 1
            g["label"] = (g["future_ret"] > 0).astype(int)
            # Drop last {forward_days} rows to avoid NaN labels
            g = g.iloc[:-forward_days]
            results.append(g)
        labeled = pd.concat(results, ignore_index=True)
        logger.info(f"Labels built. Positive rate: {labeled['label'].mean():.2%}")
        return labeled

    def prepare_xy(self, df: pd.DataFrame) -> Tuple[np.ndarray, np.ndarray]:
        """Extract features and labels from DataFrame."""
        available = [c for c in cfg.FEATURE_COLS if c in df.columns]
        X = df[available].fillna(0).values.astype(np.float32)
        y = df[cfg.TARGET_COL].values.astype(int) if cfg.TARGET_COL in df.columns else None
        return X, y

    def fit_scale(self, X_train: np.ndarray) -> np.ndarray:
        return self.scaler.fit_transform(X_train)

    def scale(self, X: np.ndarray) -> np.ndarray:
        return self.scaler.transform(X)

    def save_scaler(self):
        with open(cfg.MODEL_DIR / "scaler.pkl", "wb") as f:
            pickle.dump(self.scaler, f)
        logger.info("Scaler saved.")

    def load_scaler(self):
        with open(cfg.MODEL_DIR / "scaler.pkl", "rb") as f:
            self.scaler = pickle.load(f)
        return self

# XGBoost 

In [ ]:
class XGBoostModel:
    """GPU-accelerated XGBoost binary classifier."""

    def __init__(self):
        self.model = None
        self._load_library()

    def _load_library(self):
        try:
            import xgboost as xgb
            self._xgb = xgb
            logger.info("XGBoost loaded.")
        except ImportError:
            logger.error("xgboost not installed. Run: pip install xgboost")
            self._xgb = None

    def train(self, X_train, y_train, X_val, y_val):
        if self._xgb is None:
            return
        params = {**cfg.XGBOOST_PARAMS}
        # Fallback to CPU if GPU not available
        try:
            self.model = self._xgb.XGBClassifier(**params)
            self.model.fit(
                X_train, y_train,
                eval_set=[(X_val, y_val)],
                early_stopping_rounds=30,
                verbose=50,
            )
        except Exception as e:
            if "gpu" in str(e).lower():
                logger.warning("XGBoost GPU failed, switching to CPU.")
                params["tree_method"] = "hist"
                self.model = self._xgb.XGBClassifier(**params)
                self.model.fit(X_train, y_train, eval_set=[(X_val, y_val)],
                               early_stopping_rounds=30, verbose=50)
            else:
                raise
        logger.info("XGBoost training complete.")

    def predict_proba(self, X) -> np.ndarray:
        return self.model.predict_proba(X)[:, 1]

    def save(self):
        with open(cfg.MODEL_DIR / "xgboost.pkl", "wb") as f:
            pickle.dump(self.model, f)

    def load(self):
        with open(cfg.MODEL_DIR / "xgboost.pkl", "rb") as f:
            self.model = pickle.load(f)
        return self

# LightGBM

In [ ]:
class LGBMModel:
    """GPU-accelerated LightGBM binary classifier."""

    def __init__(self):
        self.model = None
        self._load_library()

    def _load_library(self):
        try:
            import lightgbm as lgb
            self._lgb = lgb
            logger.info("LightGBM loaded.")
        except ImportError:
            logger.error("lightgbm not installed. Run: pip install lightgbm")
            self._lgb = None

    def train(self, X_train, y_train, X_val, y_val):
        if self._lgb is None:
            return
        params = {**cfg.LGBM_PARAMS}
        try:
            self.model = self._lgb.LGBMClassifier(**params)
            self.model.fit(
                X_train, y_train,
                eval_set=[(X_val, y_val)],
                callbacks=[self._lgb.early_stopping(30), self._lgb.log_evaluation(50)],
            )
        except Exception as e:
            if "gpu" in str(e).lower():
                logger.warning("LightGBM GPU failed, switching to CPU.")
                params["device"] = "cpu"
                self.model = self._lgb.LGBMClassifier(**params)
                self.model.fit(X_train, y_train, eval_set=[(X_val, y_val)],
                               callbacks=[self._lgb.early_stopping(30)])
            else:
                raise
        logger.info("LightGBM training complete.")

    def predict_proba(self, X) -> np.ndarray:
        return self.model.predict_proba(X)[:, 1]

    def save(self):
        with open(cfg.MODEL_DIR / "lgbm.pkl", "wb") as f:
            pickle.dump(self.model, f)

    def load(self):
        with open(cfg.MODEL_DIR / "lgbm.pkl", "rb") as f:
            self.model = pickle.load(f)
        return self


# TabNet

In [ ]:
class TabNetModel:
    """
    PyTorch-TabNet for tabular classification.
    Runs on CUDA (T4 GPU). Falls back to CPU if needed.
    """

    def __init__(self):
        self.model = None
        self._load_library()

    def _load_library(self):
        try:
            from pytorch_tabnet.tab_model import TabNetClassifier
            self._TabNet = TabNetClassifier
            logger.info("PyTorch-TabNet loaded.")
        except ImportError:
            logger.error("pytorch-tabnet not installed. Run: pip install pytorch-tabnet")
            self._TabNet = None

    def train(self, X_train, y_train, X_val, y_val):
        if self._TabNet is None:
            return
        self.model = self._TabNet(
            n_d=32, n_a=32, n_steps=5,
            gamma=1.5, momentum=0.02,
            optimizer_fn=__import__("torch").optim.Adam,
            optimizer_params={"lr": 2e-3},
            scheduler_params={"step_size": 10, "gamma": 0.9},
            scheduler_fn=__import__("torch").optim.lr_scheduler.StepLR,
            mask_type="entmax",
            device_name="cuda" if __import__("torch").cuda.is_available() else "cpu",
            verbose=10,
        )
        self.model.fit(
            X_train, y_train,
            eval_set=[(X_val, y_val)],
            eval_name=["val"],
            eval_metric=["auc"],
            max_epochs=100,
            patience=20,
            batch_size=1024,
            virtual_batch_size=256,
        )
        logger.info("TabNet training complete.")

    def predict_proba(self, X) -> np.ndarray:
        preds = self.model.predict_proba(X)
        return preds[:, 1]

    def save(self):
        self.model.save_model(str(cfg.MODEL_DIR / "tabnet"))

    def load(self):
        self.model = self._TabNet()
        self.model.load_model(str(cfg.MODEL_DIR / "tabnet.zip"))
        return self


# ENSEMBLE

In [ ]:
class EnsembleModel:
    """
    Weighted ensemble of XGBoost + LightGBM + TabNet.
    Weights tuned on validation AUC.
    """

    def __init__(self):
        self.models: Dict = {}
        self.weights: Dict = cfg.ENSEMBLE_WEIGHTS
        self.data_loader = ModelDataLoader()

    def train_all(self, splits: Dict[str, pd.DataFrame]):
        X_train, y_train = self.data_loader.prepare_xy(splits["train"])
        X_val,   y_val   = self.data_loader.prepare_xy(splits["val"])

        X_train = self.data_loader.fit_scale(X_train)
        X_val   = self.data_loader.scale(X_val)

        logger.info(f"Training set: {X_train.shape} | Val set: {X_val.shape}")

        # XGBoost
        xgb = XGBoostModel()
        xgb.train(X_train, y_train, X_val, y_val)
        self.models["xgboost"] = xgb

        # LightGBM
        lgbm = LGBMModel()
        lgbm.train(X_train, y_train, X_val, y_val)
        self.models["lgbm"] = lgbm

        # TabNet
        tabnet = TabNetModel()
        tabnet.train(X_train, y_train, X_val, y_val)
        if tabnet.model is not None:
            self.models["tabnet"] = tabnet

        # Tune weights on validation
        self._tune_weights(X_val, y_val)
        self.data_loader.save_scaler()
        self.save_all()

    def _tune_weights(self, X_val, y_val):
        """Simple grid search over weight combinations."""
        from itertools import product
        best_auc, best_w = 0, dict(self.weights)
        candidates = [0.1, 0.2, 0.3, 0.4, 0.5]

        for w_xgb, w_lgbm in product(candidates, candidates):
            w_tab = round(1.0 - w_xgb - w_lgbm, 2)
            if not (0 <= w_tab <= 1):
                continue
            preds = self._weighted_predict(X_val, {"xgboost": w_xgb, "lgbm": w_lgbm, "tabnet": w_tab})
            try:
                auc = roc_auc_score(y_val, preds)
                if auc > best_auc:
                    best_auc = auc
                    best_w = {"xgboost": w_xgb, "lgbm": w_lgbm, "tabnet": w_tab}
            except Exception:
                pass

        self.weights = best_w
        logger.info(f"Tuned ensemble weights: {self.weights} (Val AUC: {best_auc:.4f})")

    def _weighted_predict(self, X, weights: Dict) -> np.ndarray:
        total, preds = 0, np.zeros(len(X))
        for name, model in self.models.items():
            if model.model is None:
                continue
            w = weights.get(name, 0.0)
            preds += w * model.predict_proba(X)
            total += w
        return preds / (total + 1e-9)

    def predict(self, X: np.ndarray) -> np.ndarray:
        X_scaled = self.data_loader.scale(X)
        return self._weighted_predict(X_scaled, self.weights)

    def evaluate(self, splits: Dict, split_name: str = "test") -> Dict:
        split = splits[split_name]
        X, y  = self.data_loader.prepare_xy(split)
        X_sc  = self.data_loader.scale(X)

        proba = self._weighted_predict(X_sc, self.weights)
        pred  = (proba >= 0.5).astype(int)

        auc = roc_auc_score(y, proba) if len(np.unique(y)) > 1 else 0.0
        report = classification_report(y, pred, output_dict=True)

        metrics = {
            "split":     split_name,
            "auc":       round(auc, 4),
            "accuracy":  round(report["accuracy"], 4),
            "precision": round(report.get("1", {}).get("precision", 0), 4),
            "recall":    round(report.get("1", {}).get("recall", 0), 4),
            "f1":        round(report.get("1", {}).get("f1-score", 0), 4),
            "n_samples": len(X),
        }
        logger.info(f"[{split_name.upper()}] AUC: {metrics['auc']} | Acc: {metrics['accuracy']} | F1: {metrics['f1']}")
        return metrics

    def save_all(self):
        for name, model in self.models.items():
            try:
                model.save()
                logger.info(f"Saved: {name}")
            except Exception as e:
                logger.warning(f"Could not save {name}: {e}")

    def load_all(self):
        for name, cls in [("xgboost", XGBoostModel), ("lgbm", LGBMModel), ("tabnet", TabNetModel)]:
            try:
                m = cls().load()
                self.models[name] = m
                logger.info(f"Loaded: {name}")
            except Exception as e:
                logger.warning(f"Could not load {name}: {e}")
        self.data_loader.load_scaler()
        return self


# BACKTEST INTEGRATION

In [ ]:
class PretrainedBacktester:
    """
    Evaluates the ensemble model on the test split using the
    event-driven backtesting engine from File 4's EventQueue.
    Measures: Total Return, Win Rate, Max Drawdown, Sharpe Ratio.
    """

    def __init__(self, model: EnsembleModel, initial_capital: float = 100_000_000):
        """initial_capital in IDR (100M IDR ~ $6,500)."""
        self.model   = model
        self.capital = initial_capital
        self.results: List[Dict] = []

    def run(self, test_df: pd.DataFrame, ohlcv: pd.DataFrame) -> Dict:
        """
        Simulate trading based on model signals on test data.
        Uses next-day close as execution price (realistic assumption).
        """
        X_test, _ = self.model.data_loader.prepare_xy(test_df)
        proba      = self.model.predict(X_test)
        test_df    = test_df.copy()
        test_df["model_score"] = proba
        test_df["signal"]      = (proba >= 0.5).astype(int)

        ohlcv_idx = ohlcv.set_index(["date", "ticker"]) if "ticker" in ohlcv.columns else ohlcv

        portfolio_value = self.capital
        positions: Dict[str, Dict] = {}
        daily_returns = []
        trades = []

        for _, row in test_df.sort_values("date").iterrows():
            ticker = row["ticker"]
            date   = pd.Timestamp(row["date"])
            signal = int(row["signal"])

            try:
                ohlcv_row = ohlcv_idx.loc[(date, ticker)]
                exec_price = float(ohlcv_row.get("adj_close", 0))
            except Exception:
                continue

            if exec_price <= 0:
                continue

            # Apply slippage
            buy_price  = exec_price * (1 + cfg.SLIPPAGE_PCT)
            sell_price = exec_price * (1 - cfg.SLIPPAGE_PCT)
            commission = exec_price * cfg.COMMISSION_PCT

            if signal == 1 and ticker not in positions:
                # BUY: allocate 2% of capital per position
                alloc = portfolio_value * 0.02
                shares = int(alloc / (buy_price + commission))
                if shares > 0:
                    cost = shares * (buy_price + commission)
                    positions[ticker] = {"shares": shares, "entry": buy_price, "date": date}
                    portfolio_value -= cost
                    trades.append({"date": date, "ticker": ticker, "action": "BUY",
                                   "price": buy_price, "shares": shares})

            elif signal == 0 and ticker in positions:
                # SELL
                pos     = positions.pop(ticker)
                revenue = pos["shares"] * (sell_price - commission)
                pnl     = revenue - pos["shares"] * pos["entry"]
                portfolio_value += revenue
                trades.append({"date": date, "ticker": ticker, "action": "SELL",
                               "price": sell_price, "shares": pos["shares"], "pnl": pnl})

        # Mark-to-market open positions at end
        for ticker, pos in positions.items():
            portfolio_value += pos["shares"] * pos["entry"]  # conservative

        # Compute metrics
        trade_df = pd.DataFrame(trades)
        metrics  = self._compute_metrics(trade_df, portfolio_value)
        logger.info(f"Backtest complete: {metrics}")
        return {"metrics": metrics, "trades": trade_df}

    def _compute_metrics(self, trades: pd.DataFrame, final_capital: float) -> Dict:
        total_return = (final_capital - self.capital) / self.capital

        sells   = trades[trades["action"] == "SELL"] if not trades.empty else pd.DataFrame()
        wins    = sells[sells.get("pnl", 0) > 0] if "pnl" in sells.columns else pd.DataFrame()
        win_rate = len(wins) / len(sells) if len(sells) > 0 else 0

        # Compute running PnL for Sharpe & Drawdown
        pnl_series = sells["pnl"].values if "pnl" in sells.columns else np.array([])
        if len(pnl_series) > 0:
            cumulative = np.cumsum(pnl_series)
            running_max = np.maximum.accumulate(cumulative)
            drawdowns   = (cumulative - running_max) / (running_max + 1e-9)
            max_drawdown = float(drawdowns.min())
            daily_r      = pnl_series / self.capital
            sharpe = (np.mean(daily_r) / (np.std(daily_r) + 1e-9)) * np.sqrt(252)
        else:
            max_drawdown, sharpe = 0.0, 0.0

        return {
            "total_return_pct":  round(total_return * 100, 2),
            "win_rate":          round(win_rate, 4),
            "max_drawdown_pct":  round(max_drawdown * 100, 2),
            "sharpe_ratio":      round(sharpe, 4),
            "total_trades":      len(trades),
            "final_capital_idr": round(final_capital, 2),
        }


In [ ]:
# MAIN
if __name__ == "__main__":
    import argparse
    parser = argparse.ArgumentParser()
    parser.add_argument("--mode", choices=["train", "eval", "inference"], default="train")
    args = parser.parse_args()

    data_loader = ModelDataLoader()

    # Try to load pipeline-produced dataset
    dataset_path = cfg.SPLIT_DIR / "model_dataset.parquet"
    splits = data_loader.load_splits(dataset_path if dataset_path.exists() else None)

    ensemble = EnsembleModel()
    ensemble.data_loader = data_loader

    if args.mode == "train":
        ensemble.train_all(splits)
        val_metrics  = ensemble.evaluate(splits, "val")
        test_metrics = ensemble.evaluate(splits, "test")
        print(f"\n✅ Val  Metrics: {val_metrics}")
        print(f"✅ Test Metrics: {test_metrics}")

        # Backtest on test split
        ohlcv = pd.read_parquet(cfg.PROC_DIR / "ohlcv_processed.parquet")
        backtester = PretrainedBacktester(ensemble)
        bt_results = backtester.run(splits["test"], ohlcv)
        print(f"\n📊 Backtest Results: {bt_results['metrics']}")

    elif args.mode == "eval":
        ensemble.load_all()
        metrics = ensemble.evaluate(splits, "test")
        print(f"Test Metrics: {metrics}")

    elif args.mode == "inference":
        ensemble.load_all()
        X_test, _ = data_loader.prepare_xy(splits["test"])
        proba = ensemble.predict(X_test)
        print(f"Inference complete. Mean score: {proba.mean():.4f}")